In [17]:
import pandas as pd

df = pd.read_csv("C:/Users/bianc/OneDrive/Documents/1-Estudos/2-DNC/1-Data Science/1-Material/Materia7-EstatisticaComPython/IA/IA em análise de Dados/Salary_dataset.csv")
df = df.drop(columns=["Unnamed: 0"])  # índice vazando

for col in ["YearsExperience", "Salary"]:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = df[(df[col] < lower) | (df[col] > upper)]

    print(f"\nColuna: {col}")
    print(f"Limites: [{lower:.2f}, {upper:.2f}]")
    print("Outliers encontrados:")
    print(outliers if not outliers.empty else "Nenhum")



Coluna: YearsExperience
Limites: [-3.45, 14.55]
Outliers encontrados:
Nenhum

Coluna: Salary
Limites: [-9014.25, 166281.75]
Outliers encontrados:
Nenhum


In [18]:
import statsmodels.api as sm

# 3. Definir X e y corretamente
X = df[["YearsExperience"]]  # só 1 feature
y = df["Salary"]

model = sm.OLS(y, X).fit()
influence = model.get_influence()

df["resid_studentized"] = influence.resid_studentized_internal
df["cooks_distance"] = influence.cooks_distance[0]

# Pontos com |resíduo studentizado| > 2
outliers_reg = df[df["resid_studentized"].abs() > 2]
outliers_reg


,YearsExperience,Salary,resid_studentized,cooks_distance
1,1.4,46206.0,2.164536,0.008268


In [19]:
from sklearn.linear_model import LinearRegression

# Criar e ajustar o modelo
model = LinearRegression()
model.fit(X, y)

# Coeficientes
coeficiente = model.coef_[0]
intercepto = model.intercept_

print("Intercepto:", intercepto)
print("Coeficiente (YearsExperience):", coeficiente)

# Qualidade do ajuste (R²)
r2 = model.score(X, y)
print("R²:", r2)


Intercepto: 24848.203966523208
Coeficiente (YearsExperience): 9449.962321455074
R²: 0.9569566641435086


In [20]:
from sklearn.metrics import mean_squared_error
import math

y_pred = model.predict(X)
rmse = math.sqrt(mean_squared_error(y, y_pred))
print("RMSE:", rmse)


RMSE: 5592.043608760661


In [21]:
# 7. Fazer previsões para 3, 5 e 10 anos de experiência
anos = [[3], [5], [10]]   # agora bate com o shape de X (uma coluna só)
predicoes = model.predict(anos)

for a, p in zip([3, 5, 10], predicoes):
    print(f"{a} anos de experiência -> salário estimado: {p:.2f}")


3 anos de experiência -> salário estimado: 53198.09
5 anos de experiência -> salário estimado: 72098.02
10 anos de experiência -> salário estimado: 119347.83


d:\Programas\Python\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


In [22]:

anos = [[22]]  # 22 anos de experiência
salario_previsto = model.predict(anos)[0]

print(f"22 anos de experiência -> salário estimado: {salario_previsto:.2f}")

22 anos de experiência -> salário estimado: 232747.38


d:\Programas\Python\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
